# L13c: Introduction to Policy Gradient and Actor-Critic Methods
In this lecture, we will explore the fundamentals of policy gradient methods and actor-critic algorithms in reinforcement learning. These methods are used for training agents to make decisions in complex environments, especially when dealing with continuous state and action spaces.

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> 
> Three learners objectives here.

Let's get started!
___

## Review: Traditional Q-Learning Problem
Q-learning iteratively estimates the state action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{\underbrace{Q_{t}(s,a)}_{\text{old value}}}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new value}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

### Algorithm
Initialize $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha_{t}$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$, the maximum number of iterations $\texttt{maxiter}$, and the convergence tolerance $\delta$. Set $\texttt{converged}\gets\texttt{false}$. 

For $s\in\mathcal{S}$
1. Initialize the trial counter $t\gets{1}$
2. While $\texttt{converged} $ is $\texttt{false}$ __do__:
    1. Roll a random number $p\in[0,1]$. Compute $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$ where $K=|\mathcal{A}|$ is the number of actions.
    2. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$.
    3. Take action $a_{t}$, observe the reward $r$ from the __world__ and transition to the next state $s^{\prime}$.
    4. Update the state-action-value function: $Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a)\right)}_{\text{new information}}$.
    5. Update the state $s\leftarrow{s^{\prime}}$, the learning rate $\alpha_{t+1}\leftarrow\alpha_{t}$, and the counter $t\leftarrow{t+1}$
    6. Convergence check: If $Q(s,a)$ has bounded change $\lVert{Q_{t+1}(s,a) - Q_{t}(s,a)}\rVert\leq\delta$, then the algorithm has converged. Set $\texttt{converged}\gets\texttt{true}$.
    7. Otherwise: if $t\geq\texttt{maxiter}$, then set $\texttt{converged}\gets\texttt{true}$ and notify the caller that the maximum iteration limit was reached without convergence. Proceed to next state.
    8. Otherwise: continue to the next iteration.
3. End While
4. End For

### Convergence
Q-learning converges to the optimal policy under two key theoretical conditions (assuming the Markov property holds for the environment):
* __Learning rate decay__: The learning rate $\alpha_{t}$ must satisfy $\sum_{t=0}^\infty \alpha_t(s, a) = \infty$ and $\sum_{t=0}^\infty \alpha_t^2(s, a) < \infty$ for all state-action pairs, ensuring sufficient initial updates while stabilizing over time. Setting $\alpha_{t+1} \gets \beta\alpha_{t}$ where $\beta<1$ is a common choice.
* __Infinite exploration__: All state-action pairs must be visited infinitely often. This condition holds for $\epsilon$-greedy policies with persistent exploration, i.e., $\epsilon_{t} > 0\,\,\forall{t}$.
___

<div>
    <center>
        <img src="figs/Q-Learning-vs-Deep-Q-Learning.ppm.png" width="580"/>
    </center>
</div>

## Deep Q-Learning Networks (DQN)
In traditional Q-learning, we maintain a table of $Q(s,a)$ values for each state-action pair. However, this approach becomes impractical when dealing with large or continuous state and action spaces. Deep Q-Learning Networks (DQN) address this limitation by using __deep neural networks__ to approximate the Q-function.

> __What is a deep neural network?__ A deep neural network is a type of artificial neural network with multiple layers between the input and output layers. These networks can learn complex patterns in data by adjusting weights through backpropagation during training. We'll deep a deep dive into neural networks in later lectures. However, for now, think of them as powerful function approximators (with a huge number of parameters) that can learn to map inputs (states) to outputs (Q-values for actions).

This approach allows for the handling of high-dimensional state spaces, such as images or continuous states, where traditional Q-learning would be infeasible due to the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality).
* __Key difference__? In this approach, the Q-value function $Q(s, a)$ is represented as a neural network, which takes the state $s$ as input and outputs the Q-values for all possible actions. The neural network is trained using the same Q-learning update rule, but with mini-batches of experiences sampled from a replay buffer to stabilize training.
* __Games__? This approach was made famous by [the DeepMind team in 2015](https://www.nature.com/articles/nature14236), where they used DQN to play Atari games directly from pixels. This approach achieved human-level performance on other games. For example, DQN was used as part of the policy network [pre-training phase for AlphaGo](https://doi.org/10.1038/nature16961), the first AI to defeat a professional human Go player, marking a milestone in AI. 
* __Other applications__? DQN has been used in other applications such as operations management, e.g., a [DQN-based system was deployed in Google’s data centers to optimize cooling, achieving a reported 30% reduction in energy consumption for cooling systems](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/) or [traffic signal control in smart cities](https://dl.acm.org/doi/10.1145/3219819.3220096), where DQN was used to optimize traffic light timings in real-time, leading to reduced congestion and improved traffic flow.

### DQN Theory
A deep Q-learning agent learns a policy $\pi$ that maximizes the expected cumulative reward $R_t$ over time. Suppose the agent is tasked with making decisions over $T\rightarrow\infty$ steps.

For each epsiode, we sample for $t = 1,2,\ldots,T$: 

1. __Interaction with the environment__: At each time step $t$, the agent observes the current state $s_t$, selects an action $a_t$ (typically using an $\epsilon$-greedy policy based on the _Q-network_), and receives a reward $r_t$ and the next state $s_{t+1}$ from the environment.
2. __Experience replay__: Each transition tuple $(s_t, a_t, r_t, s_{t+1})$ is stored in a **replay buffer** (a finite-sized memory that we'll use for training). Instead of training on consecutive samples, the agent **samples random mini-batches** from this buffer. 
3. __Main Q-Network (Function Approximator)__: The core of DQN is a deep neural network $Q_{\theta}(s)$ with (trainable) parameters $\theta$, which learns to approximate the optimal action-value function. The network takes a state as input and outputs Q-values for all possible actions.
4. __Target Q-Network__: To stabilize training, DQN uses a **target network** $Q^{\prime}_{\theta^{-}}(s)$, which is a delayed copy of the main Q-network. The target network’s parameters $\theta^-$ are updated periodically (e.g., every $N$ steps) by copying the weights from the main Q-network.

#### Batch DQN Algorithm

__Initialize__ the parameters of the main Q-network $Q_{\theta}(s)$ and the target Q-network $Q^{\prime}_{\theta^{-}}(s)$ to random values. Initialize a (potentially infinite) replay buffer $\mathcal{B}$. Set the hyperparameters: the learning rate $\alpha$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$, the minimum number of experiences in the replay buffer $B$, and the parameter update count $\mathcal{C}$.
- For each episode, initialize the state to $s_0$ and:
   - For each time step $t=1,\ldots,T$:
        1. Role a random number $p\in[0,1]$. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{\theta}(s_{t})}$.
        2. Execute action $a_{t}$, observe the reward $r_{t}$ from the _world_ and transition to the next state $s_{t+1}$. 
        3. Store the transition (experience) $\mathcal{e}=(s_t, a_t, r_t, s_{t+1})$ in the replay buffer: $\mathcal{e}\rightarrow\mathcal{B}$. 
        5. If the replay buffer $\mathcal{B}$ has a _minium number of elements_: sample a mini-batch of experiances $(s_i, a_i, r_i, s_{i+1})$ from the replay buffer.  The agent randomly samples a mini-batch of $B$ transitions from the replay buffer:  $(s_j, a_j, r_j, s_{j+1}),\, j = 1, 2, \dots, B$. Each tuple represents a state-action-reward-next state experience example collected during environment interaction.
        6. Compute the _target Q-value_ for each transition in the mini-batch using the _target Q-network_: $y_i = r_i + \gamma \cdot \max_{a^{\prime}\in\mathcal{A}}Q^{\prime}_{\theta^{-}}(s_{i+1})$ for $i=1,2,\ldots,B$.
        7. Compute the _mean squared loss_ function over the $B$ experiances collected in the mini-batch: $L(\theta) = \frac{1}{B}\sum_{i=1}^{B}\left(y_i - Q_{\theta}(s_i)\right)^2$.
        8. Perform a _single_ gradient descent step to minimize the loss function $L(\theta)$ with respect to the parameters $\theta$ of the main Q-network $Q_{\theta}(s)$: $\theta \leftarrow \theta - \alpha \nabla_{\theta}L(\theta)$, where $\alpha$ is the learning rate. 
            - _Why only a single step_? Each mini-batch is just a _small sample of the environment’s dynamics._ The goal of DQN is _online learning_: the network parameters are continuously updated as new experiences come in. If we force training to converge on each mini-batch, it risks _overfitting to that mini-batch_.
        10. Update the state $s_t \leftarrow s_{t+1}$.
        9. Every $C$ steps, update the target Q-network parameters: $\theta^{-} \leftarrow \theta$.
    - End For
- End For
___

## Policy Gradient Methods

So far we have looked at **value–based** methods, such as Q–learning and Deep Q–learning, which learn a value function $Q(s,a)$ and derive a policy from it.
In both cases, the **policy** is derived *from* the learned value function by taking an $\arg\max_a Q(s,a)$ step (possibly with exploration noise).

Policy gradient methods flip this around:

> Instead of learning $Q(s,a)$ and then extracting a policy, we directly **parameterize the policy** and choose the parameters that maximize expected return.

This gives us a clean way to handle **stochastic** policies and **continuous** action spaces where “$\max_a Q(s,a)$” is awkward or expensive.

---

### Parameterized policies

We assume the agent uses a stochastic policy
$$
\pi_\theta(a\mid s) = \mathbb{P}(A_t = a \mid S_t = s;,\theta),
$$
where $\theta$ is a vector of parameters (weights of a neural network, for example).

For a discrete action space, a common choice is a softmax policy:
$$
\pi_\theta(a\mid s) =
\frac{\exp\big(\theta_a^\top \phi(s)\big)}
{\sum_{a'} \exp\big(\theta_{a'}^\top \phi(s)\big)},
$$
where $\phi(s)$ is a feature vector of the state, and each action $a$ has its own parameter vector $\theta_a$.

Our objective is to choose $\theta$ to maximize the expected discounted return under this policy:
$$
J(\theta) = \mathbb{E}*{\pi*\theta}\big[ G_0 \big],
$$
where
$$
G_t = \sum_{k=t}^{T-1} \gamma^{,k-t} R_{k+1}
$$
is the discounted return from time $t$ and $\gamma\in[0,1)$ is the discount factor.

So the learning problem becomes a continuous optimization problem:
$$
\max_{\theta} J(\theta).
$$

---

### Policy gradient theorem (high level)

We want to compute the gradient $\nabla_\theta J(\theta)$ so we can use (stochastic) gradient ascent.

Very briefly, if we write a whole trajectory as
$$
\tau = (s_0,a_0,r_1,s_1,a_1,r_2,\dots,s_{T-1},a_{T-1},r_T),
$$
then the objective can be written as
$$
J(\theta) = \sum_{\tau} p_\theta(\tau), G(\tau),
$$
where $p_\theta(\tau)$ is the probability of trajectory $\tau$ under policy $\pi_\theta$.

Using the log–derivative trick
$$
\nabla_\theta p_\theta(\tau) = p_\theta(\tau),\nabla_\theta \log p_\theta(\tau),
$$
and the fact that only the policy depends on $\theta$, one can show that
$$
\nabla_\theta J(\theta)
= \mathbb{E}*{\pi*\theta}\left[
\sum_{t=0}^{T-1}
\nabla_\theta \log \pi_\theta(A_t\mid S_t), G_t
\right].
$$

This result is often called the **policy gradient theorem** (episodic form).

The key takeaway:

> The gradient of the expected return can be written as an expectation of $\nabla_\theta \log \pi_\theta(A_t\mid S_t)$ multiplied by a return term.

This expectation can be approximated by Monte Carlo sampling from the current policy $\pi_\theta$.

---

### REINFORCE: a Monte Carlo policy gradient algorithm

The simplest policy–gradient algorithm is called **REINFORCE**.

1. Initialize policy parameters $\theta$ (e.g., small random values).
2. For each episode:

   * Generate an episode by following $\pi_\theta$:
     $S_0,A_0,R_1,S_1,\dots,S_{T-1},A_{T-1},R_T$.
   * For each time step $t$ in the episode:

     * Compute the return from time $t$:
       $$
       G_t = \sum_{k=t}^{T-1} \gamma^{,k-t} R_{k+1}.
       $$
     * Update the policy parameters:
       $$
       \theta \leftarrow \theta

       * \alpha ,\nabla_\theta \log \pi_\theta(A_t\mid S_t), G_t,
         $$
         where $\alpha > 0$ is the learning rate.

Intuition:

* $\nabla_\theta \log \pi_\theta(A_t\mid S_t)$ tells us how to nudge the parameters to **increase** the probability of taking action $A_t$ in state $S_t$.
* Multiplying by $G_t$ means we reinforce actions that lead to high return and weaken those that lead to low return.

REINFORCE is:

* **On–policy**: episodes are generated by the current policy $\pi_\theta$.
* **Monte Carlo**: it uses complete returns, so it requires full episodes.
* **Unbiased**: the gradient estimate is correct in expectation, but it can have **high variance**.

---

### Baselines and advantages

To reduce variance, we often subtract a **baseline** $b(S_t)$ from the return:
$$
\theta \leftarrow \theta

* \alpha ,\nabla_\theta \log \pi_\theta(A_t\mid S_t),
  \big( G_t - b(S_t) \big).
  $$

If $b(S_t)$ does **not** depend on the action, this does not change the expected gradient, but it can significantly reduce variance.

A particularly useful choice is to let
$$
b(S_t) \approx V^{\pi_\theta}(S_t),
$$
in which case $G_t - b(S_t)$ is an estimate of the **advantage**
$$
A^{\pi_\theta}(S_t,A_t)
= Q^{\pi_\theta}(S_t,A_t) - V^{\pi_\theta}(S_t).
$$

Interpretation:

* $G_t - b(S_t)$ tells us whether action $A_t$ was **better or worse than average** in state $S_t$.
* The policy is updated to increase the probability of actions with positive advantage and decrease the probability of actions with negative advantage.

---

### Actor–critic (preview)

REINFORCE with a baseline suggests a natural extension:

* Use a **critic** to learn an approximation of the value function $V_w(s)$ with parameters $w$.
* Use that critic to construct a low–variance estimate of the advantage and drive the **actor** updates.

A simple actor–critic scheme:

* Critic update (temporal–difference learning):
  $$
  \delta_t = R_{t+1} + \gamma V_w(S_{t+1}) - V_w(S_t),
  $$
  $$
  w \leftarrow w + \beta, \delta_t ,\nabla_w V_w(S_t),
  $$
  where $\beta$ is the critic learning rate.

* Actor update:
  $$
  \theta \leftarrow \theta

  * \alpha, \delta_t, \nabla_\theta \log \pi_\theta(A_t\mid S_t).
    $$

Here the TD error $\delta_t$ acts as an online estimate of the advantage.

In deep RL, the actor and critic are typically implemented as neural networks:

* **Actor network:** $s \mapsto \pi_\theta(\cdot\mid s)$ (action probabilities or parameters of a continuous distribution).
* **Critic network:** $s \mapsto V_w(s)$ (scalar value estimate).

---

### Summary: value–based vs policy–based

* **Q–learning / Deep Q–learning**

  * Learn $Q(s,a)$ (table or neural network).
  * Policy is greedy or $\epsilon$–greedy w.r.t. $Q$.
  * Works best in discrete action spaces.

* **Policy gradient / actor–critic**

  * Directly parameterize $\pi_\theta(a\mid s)$.
  * Optimize expected return $J(\theta)$ via gradient ascent.
  * Naturally handles stochastic and continuous actions and connects directly to gradient–based optimization in deep learning.

Policy gradient methods and actor–critic algorithms are the foundation of many modern RL methods (e.g., A2C/A3C, DDPG, PPO), and provide a complementary perspective to the value–based Q–learning family.

## Summary
One direct, concise summary sentence goes here.

> __Key Takeaways:__
>
> Three key takeaways here.

One direct, concise conclusion sentence goes here.
___